In [1]:
import numpy as np
import pandas as pd

In [16]:
# 1. Load the Data

df = pd.read_csv('data/sensor_data.csv')

#see the column types
print(df.dtypes)
print(df.head())

#changing timestamp from str to datetime64
df = df.astype({'timestamp': 'datetime64[ns]',
                })

#multiindex
df = df.set_index(['sensor_id', 'timestamp'])

timestamp          str
sensor_id          str
temperature    float64
humidity       float64
dtype: object
             timestamp sensor_id  temperature  humidity
0  2025-10-30 08:00:00        S1        22.51     46.67
1  2025-10-30 08:05:00        S1        22.13     43.08
2  2025-10-30 08:10:00        S1        22.63     43.77
3  2025-10-30 08:15:00        S1        22.20     44.36
4  2025-10-30 08:20:00        S1        22.66     47.45


In [18]:
# Data Cleaning

temp_perc20 = df['temperature'].quantile(q=0.20)
temp_perc80 = df['temperature'].quantile(q=0.80)

#drop values
outlier = (df['temperature'] < temp_perc20) | (df['temperature'] > temp_perc80)
is_na = df.isna().any(axis=1)

rows1 = len(df)
df = df[~(outlier & is_na)]
rows2 = len(df)

#umber of rows
rows3 = rows1 - rows2
print(f'Removed {rows3} rows.')


Removed 495 rows.


In [23]:
# Filtering and Feature Engineering

# number of temp > 25 and humidity < 40
num = len(df[(df['temperature']>25) & (df['humidity'] < 40)])
print(num)

#new col
df['temp_lvl'] = np.select([df['temperature'] < 20, (df['temperature']<=30)&(df['temperature']>=20), df['temperature']>30 ], ['low', 'medium', 'high'], default = 'unknown')

#drop values
temp_mean = df['temperature'].mean()
temp_std = df['temperature'].std()

is_s5 = df.index.get_level_values('sensor_id') == 'S5'
outliers = (df['temperature'] < (temp_mean - temp_std)) | (df['temperature'] > (temp_mean + temp_std))
df = df[~(is_s5 & outliers)]


0


In [ ]:
# Time and Date Analysis
timestamps = df.index.get_level_values('timestamp')
time_diff = timestamps - timestamps.min()
df['hour_from_start'] = time_diff / pd.Timedelta(hours=1)

hours = df.index.get_level_values('timestamp').hour
df['is_daytime'] = (hours >= 6) & (hours < 18)

df['after_12h'] = (df['hour_from_start'] >= 12)

days = df.index.get_level_values('timestamp').dayofweek
df['is_weekend'] = (days >= 5)

# sensor_id == 'S2'
data_s2 = df.loc[pd.IndexSlice['S2', :], :].copy()
# rolling(6) because I have 5 minute increments
data_s2['temp_30'] = data_s2['temperature'].rolling(6).mean()
data_s2['humid_diff'] = data_s2['humidity'] - data_s2['humidity'].shift(-1)

# Have tho shift the humid_diff because the difference is computed on current-next
data_s2['temp_humid'] = (data_s2['temperature'].diff() > 5) & (data_s2['humidity'].diff() < -20)

data_s2.to_csv('data/sensor_data_S2.csv')
